# Planner enrichment for the Mujib Basin Digital Twin

This notebook builds the sub-basin planner profiles and decision-support outputs
described in Section 3.4.3 of the thesis. It integrates sub-basin geometry, suitability
rasters, NDVI summaries, ERA5 precipitation, SWAT summaries, and scenario deltas into
the JSON files loaded by the dashboard at runtime.

**Processing steps:**

1. Build the 429-sub-basin planner planning layer with indicator profiles
2. Correct suitability area accounting using exact zonal statistics
3. Merge SWAT summary statistics for the 71 overlapping sub-basins
4. Inject scenario deltas and annual scenario values from the scenario engine
5. Generate the sub-basin screening map (Vallerani-suitable, both-suitable, monitoring-needed)
6. Produce the combined suitability candidate overlay

**Inputs:** Sub-basin GeoJSON, suitability GeoTIFFs, scenario JSON, NDVI and ERA5 data

**Outputs:**
- `runtime-data/planner/subbasin_indicator_profiles_clean.json`
- `runtime-data/planner/subbasin_master_enriched_clean.json`
- `runtime-data/planner/subbasin_screening_map.geojson`

**Thesis reference:** Section 3.4.3 (methodology), Section 4.5 (screening results),
Section 4.7 (prototype outputs)


## 1. Configuration and imports


In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd

# Repository paths
REPO_ROOT = Path("..")
CESIUM_DIR = REPO_ROOT / "runtime-data"
PLANNER_DIR = REPO_ROOT / "runtime-data" / "planner"
SUIT_DIR = REPO_ROOT / "runtime-data" / "suitability"

PLANNER_DIR.mkdir(parents=True, exist_ok=True)

print("Runtime data:", CESIUM_DIR)
print("Planner output:", PLANNER_DIR)

# Output directory for intermediate files
OUT_DIR = PLANNER_DIR


## 2. Build the 429-sub-basin planner planning layer

This cell builds the enriched planner profiles from the sub-basin geometry,
suitability rasters, NDVI, and ERA5 data. Each sub-basin gets an indicator
profile containing its suitability scores, NDVI values, and precipitation context.


In [ ]:
# ======================================================================
# HELPERS
# ======================================================================
def safe_float(x: Any, default=np.nan):
    try:
        v = float(x)
        return v if np.isfinite(v) else default
    except Exception:
        return default


def find_first_column(df_or_gdf, candidates):
    lowered = {str(c).lower(): c for c in df_or_gdf.columns}
    for cand in candidates:
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    return None


def robust_normalize(series: pd.Series, pct_lo=5, pct_hi=95):
    s = pd.to_numeric(series, errors="coerce")
    s_valid = s.dropna()
    if s_valid.empty:
        return pd.Series(np.nan, index=s.index)
    lo = np.nanpercentile(s_valid, pct_lo)
    hi = np.nanpercentile(s_valid, pct_hi)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(0.0, index=s.index)
    return ((s - lo) / (hi - lo)).clip(0, 1)


def series_or_nan(df: pd.DataFrame, col: str):
    return pd.to_numeric(df[col], errors="coerce") if col in df.columns else pd.Series(np.nan, index=df.index)


def sanitize_records(records):
    out = []
    for row in records:
        clean = {}
        for k, v in row.items():
            if isinstance(v, (np.integer, int)):
                clean[k] = int(v)
            elif isinstance(v, (np.floating, float)):
                clean[k] = None if (pd.isna(v) or not np.isfinite(v)) else float(v)
            elif isinstance(v, list):
                clean[k] = [str(x) for x in v]
            else:
                clean[k] = v
        out.append(clean)
    return out




def json_safe(obj):
    """Recursively convert NaN/Inf and numpy/pandas scalars to JSON-safe Python values."""
    if obj is None or obj is pd.NA:
        return None

    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple, set)):
        return [json_safe(v) for v in obj]

    if isinstance(obj, (np.integer, int)):
        return int(obj)

    if isinstance(obj, (np.floating, float)):
        return None if (pd.isna(obj) or not np.isfinite(float(obj))) else float(obj)

    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)

    if hasattr(obj, "item") and callable(getattr(obj, "item", None)):
        try:
            return json_safe(obj.item())
        except Exception:
            pass

    return obj


def _extract_valid(arr, valid_min=None):
    vals = arr.compressed() if np.ma.isMaskedArray(arr) else np.asarray(arr).ravel()
    vals = vals[np.isfinite(vals)]
    if valid_min is not None:
        vals = vals[vals >= valid_min]
    return vals


def build_threshold_add_stats(threshold, valid_min=None):
    def count_above(arr):
        vals = _extract_valid(arr, valid_min)
        return int(np.sum(vals >= threshold)) if len(vals) else 0

    def pct_above(arr):
        vals = _extract_valid(arr, valid_min)
        return float(np.sum(vals >= threshold) / len(vals) * 100.0) if len(vals) else np.nan

    return {"count_above": count_above, "pct_above": pct_above}


def hex_to_rgb(hex_color: str):
    h = hex_color.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


def export_threshold_overlay_png(tif_path, threshold, out_png_path, color_above, valid_min, target_bounds, out_width, out_height, alpha=210):
    with rasterio.open(tif_path) as src:
        arr = src.read(1).astype(np.float32)
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)

        west, south, east, north = target_bounds
        dst_transform = from_bounds(west, south, east, north, out_width, out_height)
        dst_arr = np.full((out_height, out_width), np.nan, dtype=np.float32)

        reproject(
            source=arr,
            destination=dst_arr,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=OUT_CRS,
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.nearest,
        )

    rgba = np.zeros((out_height, out_width, 4), dtype=np.uint8)
    valid = np.isfinite(dst_arr) & (dst_arr >= valid_min)
    above = valid & (dst_arr >= threshold)
    r, g, b = hex_to_rgb(color_above)
    rgba[above, 0] = r
    rgba[above, 1] = g
    rgba[above, 2] = b
    rgba[above, 3] = alpha
    Image.fromarray(rgba, mode="RGBA").save(out_png_path)
    return {
        "path": out_png_path.name,
        "west": west,
        "south": south,
        "east": east,
        "north": north,
        "pixels_valid": int(valid.sum()),
        "pixels_above": int(above.sum()),
        "threshold": threshold,
    }


def compute_mean_annual_from_monthly(vals):
    vals = [safe_float(v, np.nan) for v in vals]
    vals = [v for v in vals if pd.notna(v)]
    if not vals:
        return np.nan
    if len(vals) <= 12:
        return float(sum(vals))
    n_years = len(vals) // 12
    if n_years == 0:
        return float(sum(vals))
    annual = [sum(vals[y*12:(y+1)*12]) for y in range(n_years)]
    return float(np.mean(annual))


def parse_era5_precip_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = {}
    if not isinstance(data, dict):
        return out

    for raw_key, value in data.items():
        key = str(raw_key).strip()
        vals = None

        if isinstance(value, dict):
            for candidate in ["p", "precip", "monthly", "values", "tp", "data"]:
                if candidate in value:
                    vals = value[candidate]
                    break
        elif isinstance(value, list):
            vals = value

        if isinstance(vals, list):
            if vals and isinstance(vals[0], dict):
                extracted = []
                for rec in vals:
                    for candidate in ["p", "precip", "tp", "value", "mm", "monthly_value"]:
                        if candidate in rec:
                            extracted.append(rec[candidate])
                            break
                vals = extracted
            out[key] = compute_mean_annual_from_monthly(vals)

    return out


def inspect_swat_parquet_compatibility(parquet_path, planning_layer_sub_ids):
    df = pd.read_parquet(parquet_path)
    sub_col = find_first_column(df, ["SUB", "sub", "Subbasin", "GRIDCODE"])
    if sub_col is None:
        raise KeyError(f"Cannot find SWAT subbasin key. Available columns: {list(df.columns)}")

    swat_sub_ids = (
        pd.to_numeric(df[sub_col], errors="coerce")
        .dropna()
        .astype("Int64")
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    swat_sub_ids = [int(x) for x in swat_sub_ids if pd.notna(x)]
    planning_layer_sub_ids = [int(x) for x in pd.Series(planning_layer_sub_ids).dropna().astype("Int64").tolist()]

    swat_set = set(swat_sub_ids)
    planning_layer_set = set(planning_layer_sub_ids)
    overlap = sorted(planning_layer_set & swat_set)
    missing_from_swat = sorted(planning_layer_set - swat_set)
    extra_in_swat = sorted(swat_set - planning_layer_set)

    return {
        "rows": int(len(df)),
        "sub_col": sub_col,
        "swat_unique_subs": int(len(swat_set)),
        "planning_layer_unique_subs": int(len(planning_layer_set)),
        "overlap_count": int(len(overlap)),
        "overlap_fraction": float(len(overlap) / len(planning_layer_set)) if planning_layer_set else np.nan,
        "exact_match": bool(swat_set == planning_layer_set),
        "missing_from_swat_count": int(len(missing_from_swat)),
        "extra_in_swat_count": int(len(extra_in_swat)),
        "sample_missing_from_swat": missing_from_swat[:15],
        "sample_extra_in_swat": extra_in_swat[:15],
        "overlap_set": set(overlap),
    }


def make_screening_recommendation(row, primary_threshold=80):
    v_mean = safe_float(row.get("mean_vallerani_suitability"), np.nan)
    m_mean = safe_float(row.get("mean_marab_suitability"), np.nan)
    v_ratio = safe_float(row.get(f"ratio_vallerani_gte{primary_threshold}"), 0.0)
    m_ratio = safe_float(row.get(f"ratio_marab_gte{primary_threshold}"), 0.0)
    ndvi_d = safe_float(row.get("mean_ndvi_delta"), np.nan)
    precip = safe_float(row.get("mean_annual_precip_mm"), np.nan)

    both_good = (v_ratio >= 0.10) and (m_ratio >= 0.05)
    v_good = (pd.notna(v_mean) and v_mean >= primary_threshold) or (v_ratio >= 0.08)
    m_good = (pd.notna(m_mean) and m_mean >= primary_threshold) or (m_ratio >= 0.08)

    if both_good:
        action = "Field-check Combined"
    elif v_good and not m_good:
        action = "Field-check Vallerani"
    elif m_good and not v_good:
        action = "Field-check Marab"
    elif v_good and m_good:
        action = "Field-check Combined"
    else:
        action = "Monitor"

    bits = []
    if pd.notna(v_mean):
        bits.append(f"Vallerani mean suitability {v_mean:.1f}%")
    if pd.notna(m_mean):
        bits.append(f"Marab mean suitability {m_mean:.1f}%")
    if pd.notna(ndvi_d):
        bits.append(f"ΔNDVI {ndvi_d:+.3f}")
    if pd.notna(precip):
        bits.append(f"ERA5 mean annual precipitation {precip:.1f} mm")

    reason = (
        "Screening-only recommendation based on suitability, NDVI, and ERA5 precipitation. "
        "SWAT, soil moisture, and MASTER logic were intentionally excluded in this 429 run."
    )
    if bits:
        reason += " Evidence: " + ", ".join(bits[:4]) + "."

    return action, "screening_only", reason

print("Helper functions defined ✅")

# ======================================================================
# CORRECTION  -  FRACTIONAL PIXEL-POLYGON SUITABILITY AREA ACCOUNTING
# ======================================================================
# This helper replaces touched-pixel counting for suitability area metrics.
# It computes:
#   - area-weighted mean suitability
#   - valid suitability area within each polygon
#   - threshold area (>= 70 / 80 / 90) using polygon ∩ pixel overlap
#
# Rationale:
# The previous workflow used rasterstats.zonal_stats(..., all_touched=True)
# with full-pixel area multiplication. That can inflate area estimates for
# small or narrow polygons and can produce impossible ratios above 1 when
# area_gte_threshold_km2 is divided by polygon area.
#
# This corrected implementation should be used for the planner suitability
# metrics that are later exported to:
#   - subbasin_master_enriched_clean.csv
#   - subbasin_master_enriched_clean.json
#   - profile / recommendation outputs derived from the enriched table
# ======================================================================

from rasterio.windows import Window as _Window
from shapely.geometry import box as _box

try:
    from shapely.validation import make_valid as _make_valid
except ImportError:
    _make_valid = None


def _fix_geom(geom):
    """Return a valid geometry, or None if it cannot be used."""
    if geom is None or geom.is_empty:
        return None
    if not geom.is_valid:
        geom = _make_valid(geom) if _make_valid else geom.buffer(0)
    if geom is None or geom.is_empty:
        return None
    return geom


def _clamp_window(src, geom_bounds):
    """Compute a raster window from polygon bounds and clamp it to raster extent."""
    from rasterio.windows import from_bounds as _win_from_bounds

    win = _win_from_bounds(*geom_bounds, transform=src.transform)
    row_off = max(0, int(np.floor(win.row_off)))
    col_off = max(0, int(np.floor(win.col_off)))
    row_end = min(src.height, int(np.ceil(win.row_off + win.height)))
    col_end = min(src.width, int(np.ceil(win.col_off + win.width)))

    if row_end <= row_off or col_end <= col_off:
        return None

    return _Window(col_off, row_off, col_end - col_off, row_end - row_off)


def fractional_suitability_stats(gdf_match, tif_path, thresholds, valid_min=None):
    """
    Compute area-weighted suitability statistics using exact polygon-pixel overlap.

    Returns one row per geometry with:
        mean
        valid_area_km2
        count_gte{T}
        pct_gte{T}
        area_gte{T}_km2
    """
    results = []

    with rasterio.open(tif_path) as src:
        if src.crs is None or getattr(src.crs, "is_geographic", False):
            raise ValueError(
                f"Raster {tif_path} must be in a projected CRS (got {src.crs}). "
                "Area calculations cannot be derived safely from geographic coordinates."
            )

        for idx, geom in enumerate(gdf_match.geometry):
            row = {"mean": np.nan, "valid_area_km2": np.nan}
            for thr in thresholds:
                row[f"count_gte{thr}"] = np.nan
                row[f"pct_gte{thr}"] = np.nan
                row[f"area_gte{thr}_km2"] = np.nan

            geom = _fix_geom(geom)
            if geom is None:
                results.append(row)
                continue

            win = _clamp_window(src, geom.bounds)
            if win is None:
                results.append(row)
                continue

            data = src.read(1, window=win, masked=True)
            transform = src.window_transform(win)

            area_valid_m2 = 0.0
            weighted_sum = 0.0
            count_above = {thr: 0 for thr in thresholds}
            area_above_m2 = {thr: 0.0 for thr in thresholds}

            for r in range(data.shape[0]):
                for c in range(data.shape[1]):
                    val = data[r, c]
                    if np.ma.is_masked(val):
                        continue

                    val = float(val)
                    if not np.isfinite(val):
                        continue
                    if valid_min is not None and val < valid_min:
                        continue

                    x1, y1 = transform * (c, r)
                    x2, y2 = transform * (c + 1, r + 1)
                    px = _box(min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2))

                    inter_area = geom.intersection(px).area
                    if inter_area <= 0:
                        continue

                    area_valid_m2 += inter_area
                    weighted_sum += val * inter_area

                    for thr in thresholds:
                        if val >= thr:
                            count_above[thr] += 1
                            area_above_m2[thr] += inter_area

            if area_valid_m2 <= 0:
                results.append(row)
                continue

            row["mean"] = weighted_sum / area_valid_m2
            row["valid_area_km2"] = area_valid_m2 / 1_000_000.0

            for thr in thresholds:
                row[f"count_gte{thr}"] = count_above[thr]
                row[f"pct_gte{thr}"] = (area_above_m2[thr] / area_valid_m2) * 100.0
                row[f"area_gte{thr}_km2"] = area_above_m2[thr] / 1_000_000.0

            results.append(row)

            if (idx + 1) % 50 == 0:
                print(f"      {idx + 1}/{len(gdf_match)} subbasins processed...")

    return pd.DataFrame(results)


## Suitability area correction

This revision corrects the suitability area accounting used in the planner enrichment workflow.

The previous implementation estimated valid suitability area and threshold area from raster cell counts with `all_touched=True`. That approach can overestimate area for small or narrow polygons because partially touched cells are counted as full cells.

The updated implementation below replaces that method with fractional pixel-polygon overlap area accounting. The exported suitability area fields and derived ratios are therefore intended to represent polygon-overlap area rather than touched-cell support.

A hard validation check is also applied before export. If any suitability threshold area still exceeds polygon area, or if any threshold ratio remains above 1, the notebook raises an error and stops instead of writing invalid outputs.


In [ ]:
# ======================================================================
# PIPELINE
# ======================================================================
print("="*70)
print("PHASE 1  -  LOAD 429 SUBBASIN PLANNING LAYER")
print("="*70)

if not SUBBASINS_GEOJSON.exists():
    raise FileNotFoundError(f"Cannot find subbasins GeoJSON: {SUBBASINS_GEOJSON}")

gdf = gpd.read_file(SUBBASINS_GEOJSON)
preferred_sub_col = find_first_column(gdf, ["Subbasin", "GRIDCODE", "SUB", "OBJECTID"])
if preferred_sub_col is None:
    raise KeyError(f"Cannot find a usable subbasin key. Available columns: {list(gdf.columns)}")

gdf = gdf.copy()
gdf["SUB"] = pd.to_numeric(gdf[preferred_sub_col], errors="coerce").astype("Int64")
missing_sub = gdf["SUB"].isna().sum()
if missing_sub:
    print(f"  ⚠ Dropping {missing_sub} features with invalid subbasin IDs")
    gdf = gdf[gdf["SUB"].notna()].copy()

gdf["era5_key"] = gdf["SUB"].astype(str)

if gdf.crs is None:
    print("  ⚠ Missing CRS  -  assuming EPSG:4326")
    gdf = gdf.set_crs("EPSG:4326")

gdf_4326 = gdf.to_crs("EPSG:4326") if gdf.crs.to_epsg() != 4326 else gdf.copy()
gdf_proj = gdf.to_crs("EPSG:32636") if gdf.crs.to_epsg() != 32636 else gdf.copy()
gdf_4326["area_km2"] = gdf_proj.geometry.area / 1e6

enriched = gdf_4326[["SUB", "era5_key", "geometry", "area_km2"]].copy()
print(f"  Planning layer key used: {preferred_sub_col}")
print(f"  Features loaded: {len(enriched)}")
print(f"  Unique SUB IDs: {enriched['SUB'].nunique()}")
print(f"  Total vector area: {enriched['area_km2'].sum():.2f} km²")

print("\n" + "="*70)
print("PHASE 2  -  ZONAL SUITABILITY")
print("="*70)
print("  Method: fractional pixel-polygon overlap area accounting")

for label, tif_path, vmin in [
    ("marab", MARAB_SUIT_TIF, MARAB_VMIN),
    ("vallerani", VALLERANI_SUIT_TIF, VALLERANI_VMIN),
]:
    print(f"\n  Processing {label}...")
    if not tif_path.exists():
        print(f"  Missing raster: {tif_path}")
        enriched[f"mean_{label}_suitability"] = np.nan
        enriched[f"valid_area_{label}_km2"] = np.nan
        for thr in THRESHOLDS:
            enriched[f"count_{label}_gte{thr}"] = np.nan
            enriched[f"pct_{label}_gte{thr}"] = np.nan
            enriched[f"area_{label}_gte{thr}_km2"] = np.nan
        continue

    with rasterio.open(tif_path) as src:
        raster_crs = src.crs

    gdf_match = gdf_4326.to_crs(raster_crs)
    stats_df = fractional_suitability_stats(
        gdf_match=gdf_match,
        tif_path=str(tif_path),
        thresholds=THRESHOLDS,
        valid_min=vmin,
    )

    enriched[f"mean_{label}_suitability"] = stats_df["mean"].values
    enriched[f"valid_area_{label}_km2"] = stats_df["valid_area_km2"].values

    for thr in THRESHOLDS:
        enriched[f"count_{label}_gte{thr}"] = stats_df[f"count_gte{thr}"].values
        enriched[f"pct_{label}_gte{thr}"] = stats_df[f"pct_gte{thr}"].values
        enriched[f"area_{label}_gte{thr}_km2"] = stats_df[f"area_gte{thr}_km2"].values

    print(f"    Mean subbasin mean: {pd.to_numeric(enriched[f'mean_{label}_suitability'], errors='coerce').mean():.2f}%")
    print(f"    Valid area total: {pd.to_numeric(enriched[f'valid_area_{label}_km2'], errors='coerce').sum():.2f} km²")
    print(f"    Area ≥{PRIMARY_THRESHOLD}%: {pd.to_numeric(enriched[f'area_{label}_gte{PRIMARY_THRESHOLD}_km2'], errors='coerce').sum():.2f} km²")

print("\n" + "="*70)
print("PHASE 3  -  NDVI ZONAL STATS")
print("="*70)
for label, tif_path in [
    ("ndvi_baseline", NDVI_BASELINE_TIF),
    ("ndvi_recent", NDVI_RECENT_TIF),
    ("ndvi_delta", NDVI_DELTA_TIF),
]:
    print(f"\n  Processing {label}...")
    if not tif_path.exists():
        print(f"  ⚠ Missing: {tif_path}")
        enriched[f"mean_{label}"] = np.nan
        enriched[f"median_{label}"] = np.nan
        continue
    with rasterio.open(tif_path) as src:
        nodata = src.nodata if src.nodata is not None else -9999
        raster_crs = src.crs
    gdf_match = gdf_4326.to_crs(raster_crs)
    stats = zonal_stats(gdf_match, str(tif_path), stats=["mean", "median", "count"], nodata=nodata, all_touched=True)
    enriched[f"mean_{label}"] = [safe_float(s.get("mean"), np.nan) for s in stats]
    enriched[f"median_{label}"] = [safe_float(s.get("median"), np.nan) for s in stats]

enriched["ndvi_trend"] = "stable"
delta = pd.to_numeric(enriched["mean_ndvi_delta"], errors="coerce")
enriched.loc[delta > 0.01, "ndvi_trend"] = "greening"
enriched.loc[delta < -0.01, "ndvi_trend"] = "declining"
print(f"  NDVI trend summary: {dict(enriched['ndvi_trend'].value_counts())}")

print("\n" + "="*70)
print("PHASE 4  -  ERA5 PRECIPITATION JOIN (ALL429)")
print("="*70)
if ERA5_PRECIP_JSON.exists():
    precip_by_sub = parse_era5_precip_json(ERA5_PRECIP_JSON)
    enriched["mean_annual_precip_mm"] = enriched["era5_key"].map(precip_by_sub)
    print(f"  Parsed ERA5 keys: {len(precip_by_sub)}")
    print(f"  Matched Mujib subbasins: {enriched['mean_annual_precip_mm'].notna().sum()}/{len(enriched)}")
else:
    print(f"  ⚠ Missing: {ERA5_PRECIP_JSON}")
    enriched["mean_annual_precip_mm"] = np.nan

print("\n" + "="*70)
print("PHASE 5  -  DISABLED MODULES FOR CLEAN 429 RUN")
print("="*70)
enriched["sm_baseline_mean"] = np.nan
enriched["sm_recent_mean"] = np.nan
enriched["sm_trend"] = "disabled_basin_only"
for c in [
    "priority_marab", "priority_vallerani", "priority_recharge", "priority_erosion",
    "best_score", "near_channel_share", "slope_pct_mean",
    "delta_runoff_pct", "delta_sediment_pct", "delta_recharge_pct",
]:
    enriched[c] = np.nan
for c in ["swat_precip_mean", "swat_runoff_mean", "swat_sediment_mean", "swat_et_mean", "swat_perc_mean", "swat_sw_mean", "swat_wyld_mean"]:
    enriched[c] = np.nan
print("  Soil moisture merge disabled: source is basin-level monthly time series, not subbasin-level.")
print("  MASTER merge disabled: source appears to be legacy 23-subbasin / what-if structure.")
print("  Scenario injection disabled: existing scenario JSON is tied to the 71-system workflow.")
print("  SWAT merge disabled until subbasin compatibility is confirmed.")

swat_overlap_set = set()
if RUN_SWAT_COMPATIBILITY_CHECK:
    print("\n" + "="*70)
    print("PHASE 6  -  SWAT COMPATIBILITY DIAGNOSTIC (NO MERGE)")
    print("="*70)
    if SWAT_PARQUET.exists():
        swat_diag = inspect_swat_parquet_compatibility(SWAT_PARQUET, enriched["SUB"])
        swat_overlap_set = swat_diag.pop("overlap_set")
        for k, v in swat_diag.items():
            print(f"  {k}: {v}")
        enriched["swat_join_candidate"] = enriched["SUB"].isin(swat_overlap_set)
    else:
        print(f"  ⚠ Missing: {SWAT_PARQUET}")
        enriched["swat_join_candidate"] = False
else:
    enriched["swat_join_candidate"] = False

print("\n" + "="*70)
print("PHASE 7  -  LIGHT SCREENING FLAGS AND RECOMMENDATIONS")
print("="*70)

for label in ["marab", "vallerani"]:
    area_col = f"area_{label}_gte{PRIMARY_THRESHOLD}_km2"
    ratio_col = f"ratio_{label}_gte{PRIMARY_THRESHOLD}"
    valid_col = f"valid_area_{label}_km2"

    enriched[ratio_col] = (
        pd.to_numeric(enriched[area_col], errors="coerce")
        / pd.to_numeric(enriched["area_km2"], errors="coerce")
    )

    enriched.loc[pd.to_numeric(enriched["area_km2"], errors="coerce") <= 0, ratio_col] = np.nan

    bad_ratio = enriched[
        pd.to_numeric(enriched[ratio_col], errors="coerce") > 1.000001
    ][["SUB", "area_km2", area_col, ratio_col]]

    if len(bad_ratio):
        raise ValueError(
            f"{ratio_col} still exceeds 1. Example rows:\n{bad_ratio.head(10).to_string(index=False)}"
        )

    bad_valid = enriched[
        pd.to_numeric(enriched[valid_col], errors="coerce")
        >  pd.to_numeric(enriched["area_km2"], errors="coerce") + 0.05
    ][["SUB", "area_km2", valid_col]]

    if len(bad_valid):
        raise ValueError(
            f"{valid_col} still exceeds polygon area. Example rows:\n{bad_valid.head(10).to_string(index=False)}"
        )

precip_series = series_or_nan(enriched, "mean_annual_precip_mm")
precip_p25 = precip_series.quantile(0.25) if precip_series.notna().any() else np.nan

all_flags = []
actions, confidences, reasons = [], [], []
for _, row in enriched.iterrows():
    flags = []
    v_mean = safe_float(row.get("mean_vallerani_suitability"), np.nan)
    m_mean = safe_float(row.get("mean_marab_suitability"), np.nan)
    v_area = safe_float(row.get(f"area_vallerani_gte{PRIMARY_THRESHOLD}_km2"), 0.0)
    m_area = safe_float(row.get(f"area_marab_gte{PRIMARY_THRESHOLD}_km2"), 0.0)
    sub_area = safe_float(row.get("area_km2"), np.nan)
    ndvi_d = safe_float(row.get("mean_ndvi_delta"), np.nan)
    precip = safe_float(row.get("mean_annual_precip_mm"), np.nan)

    if pd.notna(v_mean) and v_mean >= PRIMARY_THRESHOLD:
        flags.append(f"High Vallerani suitability ({v_mean:.1f}%)")
    if pd.notna(m_mean) and m_mean >= PRIMARY_THRESHOLD:
        flags.append(f"High Marab suitability ({m_mean:.1f}%)")
    if pd.notna(sub_area) and sub_area > 0:
        if v_area / sub_area > 0.20:
            flags.append(f"Substantial Vallerani candidate area ({v_area:.1f} km²)")
        if m_area / sub_area > 0.20:
            flags.append(f"Substantial Marab candidate area ({m_area:.1f} km²)")
    if v_area > 0 and m_area > 0:
        flags.append("Both intervention types have eligible area")
    if pd.notna(ndvi_d) and ndvi_d < -0.01:
        flags.append(f"Vegetation decline detected (ΔNDVI {ndvi_d:+.3f})")
    elif pd.notna(ndvi_d) and ndvi_d > 0.01:
        flags.append(f"Vegetation greening observed (ΔNDVI {ndvi_d:+.3f})")
    if pd.notna(precip) and pd.notna(precip_p25) and precip <= precip_p25:
        flags.append(f"Lower-quartile ERA5 precipitation ({precip:.1f} mm)")

    action, conf, reason = make_screening_recommendation(row, primary_threshold=PRIMARY_THRESHOLD)
    actions.append(action)
    confidences.append(conf)
    reasons.append(reason)
    all_flags.append(flags)

enriched["flags"] = all_flags
enriched["recommended_action"] = actions
enriched["recommendation_confidence"] = confidences
enriched["recommendation_reason"] = reasons

enriched["need_score"] = np.nan
enriched["benefit_score"] = np.nan
enriched["vallerani_feasibility"] = np.nan
enriched["marab_feasibility"] = np.nan
enriched["planner_vallerani_score"] = np.nan
enriched["planner_marab_score"] = np.nan
enriched["planner_combined_score"] = np.nan
enriched["combined_feasible"] = (
    series_or_nan(enriched, f"ratio_vallerani_gte{PRIMARY_THRESHOLD}").fillna(0) >= 0.10
) & (
    series_or_nan(enriched, f"ratio_marab_gte{PRIMARY_THRESHOLD}").fillna(0) >= 0.05
)
print(f"  Recommendation summary: {dict(enriched['recommended_action'].value_counts())}")

print("\n" + "="*70)
print("PHASE 8  -  SAVE ENRICHED MASTER OUTPUTS")
print("="*70)
csv_df = enriched.copy()
csv_df["flags"] = csv_df["flags"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
csv_df.to_csv(OUT_ENRICHED_CSV, index=False)
with open(OUT_ENRICHED_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(sanitize_records(enriched.drop(columns="geometry").to_dict(orient="records"))), f, indent=2, ensure_ascii=False, allow_nan=False)
print(f"  CSV:  {OUT_ENRICHED_CSV}")
print(f"  JSON: {OUT_ENRICHED_JSON}")

print("\n" + "="*70)
print("PHASE 9  -  BUILD PROFILES AND RECOMMENDATION JSON")
print("="*70)
profiles = {}
recommendations = {}
for _, row in enriched.iterrows():
    sub_id = int(row["SUB"])
    key = f"Subbasin_{sub_id}"
    profile = {
        "subbasin_id": sub_id,
        "era5_key": str(row.get("era5_key")),
        "area_km2": safe_float(row.get("area_km2")),
        "run_mode": {
            "planning_layer": "full_mujib_429",
            "soil_moisture_used": False,
            "master_used": False,
            "scenario_injection_used": False,
            "swat_merged": False,
            "swat_join_candidate": bool(row.get("swat_join_candidate", False)),
        },
        "condition": {
            "ndvi_baseline": safe_float(row.get("mean_ndvi_baseline")),
            "ndvi_recent": safe_float(row.get("mean_ndvi_recent")),
            "ndvi_delta": safe_float(row.get("mean_ndvi_delta")),
            "ndvi_trend": row.get("ndvi_trend", "unknown"),
            "soil_moisture_baseline": None,
            "soil_moisture_recent": None,
            "soil_moisture_trend": "disabled_basin_only",
            "mean_annual_precip_mm": safe_float(row.get("mean_annual_precip_mm")),
        },
        "suitability": {
            "vallerani_mean_pct": safe_float(row.get("mean_vallerani_suitability")),
            "marab_mean_pct": safe_float(row.get("mean_marab_suitability")),
            **{f"vallerani_area_gte{thr}_km2": safe_float(row.get(f"area_vallerani_gte{thr}_km2")) for thr in THRESHOLDS},
            **{f"marab_area_gte{thr}_km2": safe_float(row.get(f"area_marab_gte{thr}_km2")) for thr in THRESHOLDS},
            **{f"vallerani_pct_gte{thr}": safe_float(row.get(f"pct_vallerani_gte{thr}")) for thr in THRESHOLDS},
            **{f"marab_pct_gte{thr}": safe_float(row.get(f"pct_marab_gte{thr}")) for thr in THRESHOLDS},
        },
        "hydrology": {
            "era5_precip_mm": safe_float(row.get("mean_annual_precip_mm")),
            "swat_precip_mm": None,
            "swat_runoff_mm": None,
            "swat_sediment_t_ha": None,
            "swat_recharge_mm": None,
            "swat_et_mm": None,
            "swat_wyld_mm": None,
        },
        "terrain": {
            "slope_pct_mean": None,
            "near_channel_share": None,
        },
        "planner": {
            "need_score": None,
            "benefit_score": None,
            "vallerani_feasibility": None,
            "marab_feasibility": None,
            "planner_vallerani_score": None,
            "planner_marab_score": None,
            "planner_combined_score": None,
            "combined_feasible": bool(row.get("combined_feasible")) if pd.notna(row.get("combined_feasible")) else False,
            "status": "screening_only_active_layers",
        },
        "recommendation": {
            "action": row.get("recommended_action", "Monitor"),
            "confidence": row.get("recommendation_confidence", "screening_only"),
            "reason": row.get("recommendation_reason", ""),
        },
        "flags": row.get("flags", []),
    }
    profiles[key] = profile
    recommendations[key] = profile["recommendation"]
with open(OUT_PROFILES_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(profiles), f, indent=2, ensure_ascii=False, allow_nan=False)
with open(OUT_RECOMMENDATION_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(recommendations), f, indent=2, ensure_ascii=False, allow_nan=False)
print(f"  Profiles JSON:        {OUT_PROFILES_JSON}")
print(f"  Recommendation JSON: {OUT_RECOMMENDATION_JSON}")

print("\n" + "="*70)
print("PHASE 10  -  REWRITE SCENARIO-STYLE OUTPUT")
print("="*70)
if ENABLE_SCENARIO_INJECTION and SCENARIO_JSON.exists():
    with open(SCENARIO_JSON, "r", encoding="utf-8") as f:
        scenario_data = json.load(f)
    for key, profile in profiles.items():
        if key not in scenario_data:
            scenario_data[key] = {}
        scenario_data[key]["indicator_profile"] = profile
        scenario_data[key]["recommendation"] = profile["recommendation"]
        scenario_data[key]["run_mode"] = {
            "planning_layer": "full_mujib_429",
            "source_scenario_json_used": True,
            "source_scenario_json": str(SCENARIO_JSON),
            "safe_stub": False,
        }
    with open(OUT_SCENARIO_ENRICHED_JSON, "w", encoding="utf-8") as f:
        json.dump(json_safe(scenario_data), f, indent=2, ensure_ascii=False, allow_nan=False)
    print(f"  Rewrote from source scenario JSON: {OUT_SCENARIO_ENRICHED_JSON}")
elif REWRITE_SCENARIO_OUTPUT:
    scenario_stub = {}
    for key, profile in profiles.items():
        scenario_stub[key] = {
            "indicator_profile": profile,
            "recommendation": profile["recommendation"],
            "run_mode": {
                "planning_layer": "full_mujib_429",
                "source_scenario_json_used": False,
                "source_scenario_json": None,
                "safe_stub": True,
                "note": "Clean 429 scenario-style export written intentionally to overwrite stale planner output without mixing the legacy 71-system scenario source.",
            },
        }
    with open(OUT_SCENARIO_ENRICHED_JSON, "w", encoding="utf-8") as f:
        json.dump(json_safe(scenario_stub), f, indent=2, ensure_ascii=False, allow_nan=False)
    print(f"  Rewrote clean 429 scenario-style JSON: {OUT_SCENARIO_ENRICHED_JSON}")
else:
    print("  Scenario-style output rewrite disabled.")

print("\n" + "="*70)
print("PHASE 11  -  EXPORT CANDIDATE OVERLAYS")
print("="*70)
tif_files = [fp for fp in [MARAB_SUIT_TIF, VALLERANI_SUIT_TIF] if fp.exists()]
if tif_files:
    west_all, south_all = 999, 999
    east_all, north_all = -999, -999
    for fp in tif_files:
        with rasterio.open(fp) as src:
            w, s, e, n = transform_bounds(src.crs, OUT_CRS, *src.bounds, densify_pts=21)
            west_all = min(west_all, w)
            south_all = min(south_all, s)
            east_all = max(east_all, e)
            north_all = max(north_all, n)
    common_bounds = (west_all, south_all, east_all, north_all)
    manifest = {"crs": OUT_CRS, "west": west_all, "south": south_all, "east": east_all, "north": north_all, "width": OUT_WIDTH, "height": OUT_HEIGHT, "thresholds": {}}
    layer_specs = {
        "marab": {"tif": MARAB_SUIT_TIF, "color": "#FF8C00", "vmin": MARAB_VMIN},
        "vallerani": {"tif": VALLERANI_SUIT_TIF, "color": "#7B68EE", "vmin": VALLERANI_VMIN},
    }
    for thr in THRESHOLDS:
        manifest["thresholds"][str(thr)] = {}
        for label, spec in layer_specs.items():
            out_png = CANDIDATE_OUT_DIR / f"{label}_candidate_gt{thr}_4326.png"
            manifest["thresholds"][str(thr)][label] = export_threshold_overlay_png(
                tif_path=spec["tif"],
                threshold=thr,
                out_png_path=out_png,
                color_above=spec["color"],
                valid_min=spec["vmin"],
                target_bounds=common_bounds,
                out_width=OUT_WIDTH,
                out_height=OUT_HEIGHT,
                alpha=210,
            )
            print(f"  Exported: {out_png.name}")
    with open(CANDIDATE_BOUNDS_JSON, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"  Manifest JSON: {CANDIDATE_BOUNDS_JSON}")
else:
    print("  ⚠ No suitability TIFFs found  -  skipping candidate overlays")

print("\n" + "="*70)
print("PHASE 12  -  VALIDATION DIAGNOSTICS")
print("="*70)
diag = pd.DataFrame({
    "SUB": enriched["SUB"].astype("Int64"),
    "era5_key": enriched["era5_key"],
    "area_km2": enriched["area_km2"],
    "has_marab": enriched["mean_marab_suitability"].notna(),
    "has_vallerani": enriched["mean_vallerani_suitability"].notna(),
    "has_ndvi_baseline": enriched["mean_ndvi_baseline"].notna(),
    "has_ndvi_recent": enriched["mean_ndvi_recent"].notna(),
    "has_ndvi_delta": enriched["mean_ndvi_delta"].notna(),
    "has_era5_precip": enriched["mean_annual_precip_mm"].notna(),
    "swat_join_candidate": enriched["swat_join_candidate"].astype(bool),
    "recommended_action": enriched["recommended_action"],
})
diag.to_csv(OUT_VALIDATION_CSV, index=False)
print(f"  Validation CSV: {OUT_VALIDATION_CSV}")
print(f"  Active-layer complete rows: {(diag['has_marab'] & diag['has_vallerani'] & diag['has_ndvi_delta'] & diag['has_era5_precip']).sum()}/{len(diag)}")

print("\n" + "="*70)
print("PIPELINE COMPLETE")
print("="*70)
for fp in [OUT_ENRICHED_CSV, OUT_ENRICHED_JSON, OUT_PROFILES_JSON, OUT_RECOMMENDATION_JSON, OUT_VALIDATION_CSV, CANDIDATE_BOUNDS_JSON]:
    if fp.exists():
        print(f"  {fp.name}")


## Safe-overlap SWAT merge

This step merges SWAT summary statistics only for the reliable overlap subset and rewrites the same planner outputs in place. The 429 planning layer remains the primary planner structure; SWAT is added only where a safe join is available.


In [ ]:
# ==========================================================
# NEXT STEP  -  PARTIAL SWAT MERGE FOR SAFE OVERLAP ONLY (REWRITE IN PLACE)
# Keeps 429 planning_layer, keeps HTML untouched, keeps same filenames
#
# What this does:
#   1) Merges SWAT means for the safe overlap subbasins only
#   2) Rebuilds flags using suitability + NDVI + ERA5 + SWAT
#   3) Runs cross-validation diagnostics for thesis reporting
#   4) Rewrites the SAME existing planner files in place
# ==========================================================
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("PARTIAL SWAT MERGE INTO 429 PLANNING LAYER")
print("=" * 70)

if not SWAT_PARQUET.exists():
    raise FileNotFoundError(f"Missing SWAT parquet: {SWAT_PARQUET}")

df_swat = pd.read_parquet(SWAT_PARQUET)

# ----------------------------
# 1) Detect SWAT subbasin key
# ----------------------------
sub_col = None
for c in ["SUB", "sub", "Subbasin", "SUBBASIN"]:
    if c in df_swat.columns:
        sub_col = c
        break
if sub_col is None:
    raise KeyError(f"Could not find SWAT subbasin key. Available columns: {list(df_swat.columns)}")

df_swat = df_swat.copy()
df_swat["_SUB"] = pd.to_numeric(df_swat[sub_col], errors="coerce")
df_swat = df_swat[df_swat["_SUB"].notna()].copy()
df_swat["_SUB"] = df_swat["_SUB"].astype(int)

planning_layer_ids = set(pd.to_numeric(enriched["SUB"], errors="coerce").dropna().astype(int))
swat_ids = set(df_swat["_SUB"].unique().tolist())
overlap_ids = sorted(planning_layer_ids.intersection(swat_ids))

print(f"SWAT unique subbasins : {len(swat_ids)}")
print(f"429 planning_layer subbasins: {len(planning_layer_ids)}")
print(f"Safe overlap to merge : {len(overlap_ids)}")
print(f"Excluded SWAT-only IDs: {sorted(list(swat_ids - planning_layer_ids))[:20]}")

# ----------------------------
# 2) Detect SWAT metric columns robustly
# ----------------------------
alias_map = {
    "swat_precip_mean":   ["PRECIPmm", "PRECIP_MM", "PRECIP", "PCPmm", "PCP_MM", "RAINmm"],
    "swat_runoff_mean":   ["SURQmm", "SURQ_MM", "SURQ", "RUNOFFmm", "RUNOFF_MM"],
    "swat_sediment_mean": ["SYLDt_ha", "SYLD_T_HA", "SYLD", "SEDt_ha", "SEDIMENT_t_ha"],
    "swat_et_mean":       ["ETmm", "ET_MM", "ET"],
    "swat_perc_mean":     ["PERCmm", "PERC_MM", "PERC"],
    "swat_sw_mean":       ["SWmm", "SW_MM", "SW"],
    "swat_wyld_mean":     ["WYLDmm", "WYLD_MM", "WYLD", "WYLDmm_ha"],
}

resolved_cols = {}
for out_col, aliases in alias_map.items():
    found = None
    for a in aliases:
        if a in df_swat.columns:
            found = a
            break
    resolved_cols[out_col] = found

print("\nResolved SWAT columns:")
for out_col, src_col in resolved_cols.items():
    print(f"  {out_col:20} -> {src_col}")

usable = {k: v for k, v in resolved_cols.items() if v is not None}
if not usable:
    raise ValueError("No usable SWAT metric columns found in parquet.")

# Ensure target columns exist in enriched even if some are absent in parquet
for out_col in alias_map.keys():
    if out_col not in enriched.columns:
        enriched[out_col] = np.nan

# ----------------------------
# 3) Aggregate SWAT means for overlap only
# ----------------------------
df_use = df_swat[df_swat["_SUB"].isin(overlap_ids)].copy()

agg_spec = {src_col: "mean" for src_col in usable.values()}
swat_means = (
    df_use.groupby("_SUB")
    .agg(agg_spec)
    .reset_index()
    .rename(columns={"_SUB": "SUB"})
)

rename_back = {src_col: out_col for out_col, src_col in usable.items()}
swat_means = swat_means.rename(columns=rename_back)

print(f"\nRows in SWAT overlap table: {len(swat_means)}")

# ----------------------------
# 4) Merge into enriched in memory
# ----------------------------
lookup = swat_means.set_index("SUB")
sub_series = pd.to_numeric(enriched["SUB"], errors="coerce")

for out_col in usable.keys():
    enriched[out_col] = sub_series.map(lookup[out_col])

enriched["swat_join_candidate"] = enriched["SUB"].isin(overlap_ids)
enriched["has_swat"] = enriched[list(usable.keys())].notna().any(axis=1)

print(f"Rows with any SWAT data after merge: {int(enriched['has_swat'].sum())}")
print(f"Rows without SWAT data            : {int((~enriched['has_swat']).sum())}")

if enriched["has_swat"].any():
    if enriched["swat_runoff_mean"].notna().any():
        print(f"Mean runoff   (SWAT subs): {enriched.loc[enriched['has_swat'], 'swat_runoff_mean'].mean():.2f} mm")
    if enriched["swat_sediment_mean"].notna().any():
        print(f"Mean sediment (SWAT subs): {enriched.loc[enriched['has_swat'], 'swat_sediment_mean'].mean():.3f} t/ha")

# ----------------------------
# 5) Add SWAT-aware flags, keep decision-support framing
# ----------------------------
runoff_p75 = enriched["swat_runoff_mean"].dropna().quantile(0.75) if enriched["swat_runoff_mean"].notna().any() else np.nan
sed_p75 = enriched["swat_sediment_mean"].dropna().quantile(0.75) if enriched["swat_sediment_mean"].notna().any() else np.nan
precip_p25 = enriched["mean_annual_precip_mm"].dropna().quantile(0.25) if enriched["mean_annual_precip_mm"].notna().any() else np.nan

def make_flags_with_swat(row):
    flags = []

    # NDVI
    if row.get("ndvi_trend") == "declining":
        flags.append("Vegetation decline detected")
    elif row.get("ndvi_trend") == "greening":
        flags.append("Vegetation greening detected")

    # Suitability flags only when suitability is resolved
    if row.get("suitability_status") != "unresolved_in_source_raster":
        if pd.notna(row.get("area_vallerani_gte80_km2")) and row.get("area_vallerani_gte80_km2", 0) > 0:
            flags.append("Vallerani feasible area present at ≥80% threshold")
        if pd.notna(row.get("area_marab_gte80_km2")) and row.get("area_marab_gte80_km2", 0) > 0:
            flags.append("Marab feasible area present at ≥80% threshold")
    else:
        flags.append("Suitability unresolved in source raster")

    # ERA5 missing / dry-context flag
    if row.get("era5_status") == "missing_in_source_json":
        flags.append("ERA5 precipitation missing in source JSON")
    else:
        precip = row.get("mean_annual_precip_mm")
        if pd.notna(precip) and pd.notna(precip_p25) and precip <= precip_p25:
            flags.append(f"Lower-quartile ERA5 precipitation ({precip:.1f} mm)")

    # SWAT hydrology highlights where available
    runoff = row.get("swat_runoff_mean")
    sed = row.get("swat_sediment_mean")

    if pd.notna(runoff) and pd.notna(runoff_p75) and runoff >= runoff_p75:
        flags.append(f"Above-average surface runoff ({runoff:.1f} mm)")

    if pd.notna(sed) and pd.notna(sed_p75) and sed >= sed_p75:
        flags.append(f"Above-average sediment yield ({sed:.2f} t/ha)")

    return flags

enriched["flags"] = enriched.apply(make_flags_with_swat, axis=1)

def build_note_with_swat(row):
    notes = [
        "Decision-support only. This output summarizes suitability, NDVI, ERA5 precipitation, and SWAT hydrology where available for stakeholder interpretation. It does not prescribe an intervention."
    ]

    if row.get("era5_status") == "missing_in_source_json":
        notes.append("ERA5 precipitation is missing because the ALL429 source JSON does not contain this subbasin ID.")

    if row.get("suitability_status") == "unresolved_in_source_raster":
        notes.append("Suitability remains unresolved because the source suitability raster has no valid cells for this subbasin after cleaning.")

    if bool(row.get("has_swat", False)):
        notes.append("SWAT summary metrics are available for this subbasin from the overlapping modeled subset.")
    else:
        notes.append("No SWAT data are available for this subbasin in the current overlap set.")

    return " ".join(notes)

enriched["decision_support_note"] = enriched.apply(build_note_with_swat, axis=1)

# keep recommendation fields neutral
enriched["recommended_action"] = None
enriched["recommendation_confidence"] = None
enriched["recommendation_reason"] = None
enriched["output_framing"] = "decision_support_only"

# ----------------------------
# 5b) CROSS-VALIDATION DIAGNOSTICS (for thesis)
# ----------------------------
print("\n" + "=" * 70)
print("CROSS-VALIDATION DIAGNOSTICS")
print("=" * 70)

diag = pd.DataFrame({"SUB": enriched["SUB"]})

# layer presence summary
diag["has_marab"] = enriched["mean_marab_suitability"].notna()
diag["has_vallerani"] = enriched["mean_vallerani_suitability"].notna()
diag["has_ndvi_delta"] = enriched["mean_ndvi_delta"].notna()
diag["has_era5_precip"] = enriched["mean_annual_precip_mm"].notna()
diag["has_swat"] = enriched["has_swat"].astype(bool)

# Water balance: P ≈ ET + SURQ + PERC
p = pd.to_numeric(enriched.get("swat_precip_mean"), errors="coerce")
et = pd.to_numeric(enriched.get("swat_et_mean"), errors="coerce")
surq = pd.to_numeric(enriched.get("swat_runoff_mean"), errors="coerce")
perc = pd.to_numeric(enriched.get("swat_perc_mean"), errors="coerce").fillna(0)

wb_valid = p.notna() & et.notna() & surq.notna()
diag["wb_valid"] = wb_valid

if wb_valid.sum() > 0:
    residual_pct = ((p - (et + surq + perc)) / p * 100)
    diag["wb_residual_pct"] = residual_pct

    valid_res = residual_pct[wb_valid]
    print(f"\n  Water balance (n={wb_valid.sum()}):")
    print(f"    Median residual: {valid_res.median():.1f}% of precip")
    print(f"    Range: {valid_res.min():.1f}% to {valid_res.max():.1f}%")
else:
    diag["wb_residual_pct"] = np.nan
    print("\n  Water balance: insufficient SWAT variables for comparison")

# ERA5 vs SWAT precipitation
era5_p = pd.to_numeric(enriched.get("mean_annual_precip_mm"), errors="coerce")
swat_p = pd.to_numeric(enriched.get("swat_precip_mean"), errors="coerce")
both_precip = era5_p.notna() & swat_p.notna()
diag["era5_vs_swat_valid"] = both_precip
diag["era5_minus_swat_precip"] = era5_p - swat_p

if both_precip.sum() > 5:
    corr = era5_p[both_precip].corr(swat_p[both_precip])
    bias = (era5_p[both_precip] - swat_p[both_precip]).mean()
    print(f"\n  ERA5 vs SWAT precip (n={both_precip.sum()}):")
    print(f"    Pearson r = {corr:.3f}")
    print(f"    Mean bias (ERA5 − SWAT): {bias:.1f} mm")
else:
    print(f"\n  ERA5 vs SWAT precip: only {both_precip.sum()} valid pairs")

# NDVI vs SWAT ET
ndvi_b = pd.to_numeric(enriched.get("mean_ndvi_baseline"), errors="coerce")
swat_et = pd.to_numeric(enriched.get("swat_et_mean"), errors="coerce")
both_ndvi_et = ndvi_b.notna() & swat_et.notna()
diag["ndvi_vs_et_valid"] = both_ndvi_et

if both_ndvi_et.sum() > 5:
    corr = ndvi_b[both_ndvi_et].corr(swat_et[both_ndvi_et])
    print(f"\n  NDVI baseline vs SWAT ET (n={both_ndvi_et.sum()}):")
    print(f"    Pearson r = {corr:.3f} ({'consistent' if corr > 0 else 'CHECK DATA'})")
else:
    print(f"\n  NDVI baseline vs SWAT ET: only {both_ndvi_et.sum()} valid pairs")

# save same existing validation file in place
diag.to_csv(OUT_VALIDATION_CSV, index=False)
print(f"\nSaved validation diagnostics: {OUT_VALIDATION_CSV}")

# ----------------------------
# 6) Rewrite SAME files in place
# ----------------------------
csv_df = enriched.copy()
csv_df["flags"] = csv_df["flags"].apply(lambda x: "; ".join(x) if isinstance(x, list) else "")
csv_df["missing_active_fields"] = csv_df["missing_active_fields"].apply(
    lambda x: "; ".join(x) if isinstance(x, list) else ""
)
csv_df.to_csv(OUT_ENRICHED_CSV, index=False)

with open(OUT_ENRICHED_JSON, "w", encoding="utf-8") as f:
    json.dump(
        json_safe(sanitize_records(enriched.drop(columns="geometry").to_dict(orient="records"))),
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )

profiles = {}
briefs = {}
scenario_stub = {}

for _, row in enriched.iterrows():
    sid = int(row["SUB"])
    key = f"Subbasin_{sid}"

    profile = {
        "subbasin_id": sid,
        "condition": {
            "mean_annual_precip_mm": safe_float(row.get("mean_annual_precip_mm")),
            "ndvi_baseline": safe_float(row.get("mean_ndvi_baseline")),
            "ndvi_recent": safe_float(row.get("mean_ndvi_recent")),
            "ndvi_delta": safe_float(row.get("mean_ndvi_delta")),
            "ndvi_trend": row.get("ndvi_trend"),
        },
        "suitability": {
            "marab_mean_pct": safe_float(row.get("mean_marab_suitability")),
            "vallerani_mean_pct": safe_float(row.get("mean_vallerani_suitability")),
            "marab_area_gte80_km2": safe_float(row.get("area_marab_gte80_km2")),
            "vallerani_area_gte80_km2": safe_float(row.get("area_vallerani_gte80_km2")),
        },
        "hydrology": {
            "has_swat": bool(row.get("has_swat", False)),
            "swat_precip_mm": safe_float(row.get("swat_precip_mean")),
            "swat_runoff_mm": safe_float(row.get("swat_runoff_mean")),
            "swat_sediment_t_ha": safe_float(row.get("swat_sediment_mean")),
            "swat_recharge_mm": safe_float(row.get("swat_perc_mean")),
            "swat_et_mm": safe_float(row.get("swat_et_mean")),
            "swat_soil_water_mm": safe_float(row.get("swat_sw_mean")),
            "swat_water_yield_mm": safe_float(row.get("swat_wyld_mean")),
        },
        "planner": {
            "active_layers_complete": bool(row.get("active_layers_complete", False)),
            "missing_active_fields": row.get("missing_active_fields", []),
            "era5_status": row.get("era5_status"),
            "suitability_status": row.get("suitability_status"),
            "swat_join_candidate": bool(row.get("swat_join_candidate", False)),
            "output_framing": "decision_support_only",
        },
        "flags": row.get("flags", []),
        "decision_support_note": row.get("decision_support_note", ""),
    }

    profiles[key] = profile

    briefs[key] = {
        "subbasin_id": sid,
        "indicator_profile": {
            "condition": profile["condition"],
            "suitability": profile["suitability"],
            "hydrology": profile["hydrology"],
        },
        "flags": profile["flags"],
        "decision_support_note": profile["decision_support_note"],
        "planner": profile["planner"],
    }

    scenario_stub[key] = {
        "indicator_profile": profile,
        "decision_support": {
            "flags": profile["flags"],
            "note": profile["decision_support_note"],
            "framing": "decision_support_only"
        }
    }

with open(OUT_PROFILES_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(profiles), f, indent=2, ensure_ascii=False, allow_nan=False)

with open(OUT_RECOMMENDATION_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(briefs), f, indent=2, ensure_ascii=False, allow_nan=False)

with open(OUT_SCENARIO_ENRICHED_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(scenario_stub), f, indent=2, ensure_ascii=False, allow_nan=False)

print("\nRewrote same existing files in place with partial SWAT merge:")
print(" -", OUT_ENRICHED_CSV)
print(" -", OUT_ENRICHED_JSON)
print(" -", OUT_PROFILES_JSON)
print(" -", OUT_RECOMMENDATION_JSON)
print(" -", OUT_SCENARIO_ENRICHED_JSON)

print("\n" + "=" * 70)
print("ENRICHMENT SUMMARY")
print("=" * 70)
print(f"  Total subbasins:           {len(enriched)}")
print(f"  With suitability:          {enriched['mean_marab_suitability'].notna().sum()}")
print(f"  With NDVI:                 {enriched['mean_ndvi_delta'].notna().sum()}")
print(f"  With ERA5 precip:          {enriched['mean_annual_precip_mm'].notna().sum()}")
print(f"  With SWAT hydrology:       {int(enriched['has_swat'].sum())}")
print(f"  With ≥1 flag:              {sum(1 for f in enriched['flags'] if isinstance(f, list) and len(f) > 0)}")

## Scenario delta enrichment

This step injects scenario deltas and annual scenario values from the current `scenarios_USED_BY_CESIUM_FINAL_71.json` source into the planner-facing JSON outputs for subbasins with scenario availability.


In [ ]:
# ==========================================================
# FINAL STANDALONE CELL  -  WRITE SCENARIO DELTAS IN PLACE
# No dependency on earlier notebook variables
# Does NOT touch HTML
# ==========================================================
import json
import math
from pathlib import Path

print("=" * 70)
print("FINAL STANDALONE SCENARIO DELTA WRITE")
print("=" * 70)

# ----------------------------
# PATHS
# ----------------------------
BASE = REPO_ROOT
MUJIB_DT = BASE / "MUJIB_DT"
CESIUM_71 = MUJIB_DT / "Cesium_71"
PLANNER_DIR = CESIUM_71 / "planner"

SCENARIO_JSON = CESIUM_71 / "scenarios_USED_BY_CESIUM_FINAL_71.json"
OUT_MASTER_JSON = PLANNER_DIR / "subbasin_master_enriched_clean.json"
OUT_PROFILES_JSON = PLANNER_DIR / "subbasin_indicator_profiles_clean.json"
OUT_RECOMMENDATION_JSON = PLANNER_DIR / "subbasin_recommendations_clean.json"
OUT_SCENARIO_ENRICHED_JSON = PLANNER_DIR / "scenarios_ENRICHED_WITH_PROFILES_clean.json"

for fp in [
    SCENARIO_JSON,
    OUT_MASTER_JSON,
    OUT_PROFILES_JSON,
    OUT_RECOMMENDATION_JSON,
    OUT_SCENARIO_ENRICHED_JSON,
]:
    if not fp.exists():
        raise FileNotFoundError(f"Missing file: {fp}")

# ----------------------------
# LOAD FILES
# ----------------------------
with open(SCENARIO_JSON, "r", encoding="utf-8") as f:
    source_scenarios = json.load(f)

with open(OUT_MASTER_JSON, "r", encoding="utf-8") as f:
    master_rows = json.load(f)

with open(OUT_PROFILES_JSON, "r", encoding="utf-8") as f:
    profiles = json.load(f)

with open(OUT_RECOMMENDATION_JSON, "r", encoding="utf-8") as f:
    briefs = json.load(f)

with open(OUT_SCENARIO_ENRICHED_JSON, "r", encoding="utf-8") as f:
    scenario_enriched = json.load(f)

# ----------------------------
# HELPERS
# ----------------------------
def finite(x):
    try:
        return x is not None and math.isfinite(float(x))
    except Exception:
        return False

def pct_delta(base, scen):
    if not finite(base) or not finite(scen):
        return None
    base = float(base)
    scen = float(scen)
    if abs(base) < 1e-12:
        return None
    return round(((scen - base) / base) * 100.0, 1)

# ----------------------------
# EXTRACT strict default_whatif -> annual scenario blocks
# ----------------------------
scenario_delta_map = {}
scenario_annual_value_map = {}
problems = []

for sub_key, entry in source_scenarios.items():
    default_whatif = entry.get("default_whatif")
    whatifs = entry.get("whatifs", {})

    if not default_whatif or default_whatif not in whatifs:
        problems.append((sub_key, "missing default_whatif"))
        continue

    annual = whatifs[default_whatif].get("annual")
    if not isinstance(annual, dict):
        problems.append((sub_key, "missing annual block"))
        continue

    needed = ["baseline", "marab", "vallerani", "combined"]
    if not all(k in annual for k in needed):
        problems.append((sub_key, "missing one or more scenario annual blocks"))
        continue

    b = annual["baseline"]
    m = annual["marab"]
    v = annual["vallerani"]
    c = annual["combined"]

    annual_values = {
        "baseline": {
            "runoff": b.get("runoff"),
            "sediment": b.get("sediment"),
            "groundwater": b.get("groundwater"),
            "soil_moisture": b.get("soil_moisture"),
            "vegetation": b.get("vegetation"),
            "precip": b.get("precip"),
            "pet": b.get("pet"),
        },
        "marab": {
            "runoff": m.get("runoff"),
            "sediment": m.get("sediment"),
            "groundwater": m.get("groundwater"),
            "soil_moisture": m.get("soil_moisture"),
            "vegetation": m.get("vegetation"),
            "precip": m.get("precip"),
            "pet": m.get("pet"),
        },
        "vallerani": {
            "runoff": v.get("runoff"),
            "sediment": v.get("sediment"),
            "groundwater": v.get("groundwater"),
            "soil_moisture": v.get("soil_moisture"),
            "vegetation": v.get("vegetation"),
            "precip": v.get("precip"),
            "pet": v.get("pet"),
        },
        "combined": {
            "runoff": c.get("runoff"),
            "sediment": c.get("sediment"),
            "groundwater": c.get("groundwater"),
            "soil_moisture": c.get("soil_moisture"),
            "vegetation": c.get("vegetation"),
            "precip": c.get("precip"),
            "pet": c.get("pet"),
        },
    }

    deltas = {
        "marab": {
            "runoff": pct_delta(b.get("runoff"), m.get("runoff")),
            "sediment": pct_delta(b.get("sediment"), m.get("sediment")),
            "recharge": pct_delta(b.get("groundwater"), m.get("groundwater")),
        },
        "vallerani": {
            "runoff": pct_delta(b.get("runoff"), v.get("runoff")),
            "sediment": pct_delta(b.get("sediment"), v.get("sediment")),
            "recharge": pct_delta(b.get("groundwater"), v.get("groundwater")),
        },
        "combined": {
            "runoff": pct_delta(b.get("runoff"), c.get("runoff")),
            "sediment": pct_delta(b.get("sediment"), c.get("sediment")),
            "recharge": pct_delta(b.get("groundwater"), c.get("groundwater")),
        },
    }

    scenario_delta_map[sub_key] = deltas
    scenario_annual_value_map[sub_key] = annual_values

print(f"Source scenario subbasins : {len(source_scenarios)}")
print(f"Delta blocks extracted    : {len(scenario_delta_map)}")
print(f"Problems                  : {len(problems)}")
if problems:
    print("First 10 problems:")
    for p in problems[:10]:
        print(" ", p)

# ----------------------------
# INJECT INTO MASTER JSON
# master is row-list keyed by SUB
# ----------------------------
master_touched = 0
for row in master_rows:
    sub = row.get("SUB")
    try:
        sub_id = int(sub)
        key = f"Subbasin_{sub_id}"
    except Exception:
        key = None

    deltas = scenario_delta_map.get(key) if key else None
    annual_vals = scenario_annual_value_map.get(key) if key else None

    row["scenario_deltas_pct"] = deltas if deltas else None
    row["scenario_annual_values"] = annual_vals if annual_vals else None
    row["scenario_data_status"] = "available" if deltas else "not_available"
    master_touched += 1

# ----------------------------
# INJECT INTO PROFILES JSON
# ----------------------------
profiles_touched = 0
for key, profile in profiles.items():
    deltas = scenario_delta_map.get(key)
    annual_vals = scenario_annual_value_map.get(key)

    profile["scenario_deltas_pct"] = deltas if deltas else None
    profile["scenario_annual_values"] = annual_vals if annual_vals else None
    profile["scenario_data_status"] = "available" if deltas else "not_available"
    profiles_touched += 1

# ----------------------------
# INJECT INTO BRIEF JSON
# ----------------------------
briefs_touched = 0
for key, brief in briefs.items():
    deltas = scenario_delta_map.get(key)
    annual_vals = scenario_annual_value_map.get(key)

    brief["scenario_deltas_pct"] = deltas if deltas else None
    brief["scenario_annual_values"] = annual_vals if annual_vals else None
    brief["scenario_data_status"] = "available" if deltas else "not_available"
    briefs_touched += 1

# ----------------------------
# INJECT INTO SCENARIO-ENRICHED JSON
# ----------------------------
scenario_touched = 0
for key, obj in scenario_enriched.items():
    deltas = scenario_delta_map.get(key)
    annual_vals = scenario_annual_value_map.get(key)

    if "indicator_profile" in obj and isinstance(obj["indicator_profile"], dict):
        obj["indicator_profile"]["scenario_deltas_pct"] = deltas if deltas else None
        obj["indicator_profile"]["scenario_annual_values"] = annual_vals if annual_vals else None
        obj["indicator_profile"]["scenario_data_status"] = "available" if deltas else "not_available"

    if "decision_support" not in obj or not isinstance(obj["decision_support"], dict):
        obj["decision_support"] = {}

    obj["decision_support"]["scenario_deltas_pct"] = deltas if deltas else None
    obj["decision_support"]["scenario_annual_values"] = annual_vals if annual_vals else None
    obj["decision_support"]["scenario_data_status"] = "available" if deltas else "not_available"

    scenario_touched += 1

# ----------------------------
# WRITE BACK SAME FILES IN PLACE
# ----------------------------
with open(OUT_MASTER_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(master_rows), f, indent=2, ensure_ascii=False, allow_nan=False)

with open(OUT_PROFILES_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(profiles), f, indent=2, ensure_ascii=False, allow_nan=False)

with open(OUT_RECOMMENDATION_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(briefs), f, indent=2, ensure_ascii=False, allow_nan=False)

with open(OUT_SCENARIO_ENRICHED_JSON, "w", encoding="utf-8") as f:
    json.dump(json_safe(scenario_enriched), f, indent=2, ensure_ascii=False, allow_nan=False)

print("\nInjection summary:")
print(f"  Master rows touched       : {master_touched}")
print(f"  Profiles touched          : {profiles_touched}")
print(f"  Briefs touched            : {briefs_touched}")
print(f"  Scenario-enriched touched : {scenario_touched}")

# ----------------------------
# SAMPLE CHECK
# ----------------------------
sample_key = next((k for k, v in profiles.items() if v.get("scenario_deltas_pct")), None)
if sample_key:
    print("\nSample profile block:")
    print(sample_key)
    print(json.dumps({
        "hydrology": profiles[sample_key].get("hydrology"),
        "scenario_annual_values": profiles[sample_key].get("scenario_annual_values"),
        "scenario_deltas_pct": profiles[sample_key].get("scenario_deltas_pct"),
        "scenario_data_status": profiles[sample_key].get("scenario_data_status"),
    }, indent=2))

print("\nRewrote same existing files in place:")
print(" -", OUT_MASTER_JSON)
print(" -", OUT_PROFILES_JSON)
print(" -", OUT_RECOMMENDATION_JSON)
print(" -", OUT_SCENARIO_ENRICHED_JSON)

## 6. Generate sub-basin screening map

Sort the 429 planning sub-basins into four classes based on suitability scores:
primarily Vallerani-suitable, suitable for both interventions, monitoring-needed,
and excluded. This classification is reported in Section 4.5.


In [ ]:
# ==========================================================
# PHASE 13  -  EXPORT SUBBASIN SCREENING MAP (GEOJSON)
#
# Joins a rule-based screening classification back to the
# 429-subbasin polygon planning_layer and exports a single GeoJSON
# for use as a choropleth layer in the CesiumJS dashboard.
#
# Classification uses five classes based on the >= 80
# suitability planning threshold:
#   - Vallerani priority
#   - Marab priority
#   - Both feasible
#   - Monitor
#   - Unresolved
#
# Class order: Unresolved -> priority checks -> Both feasible -> Monitor.
# Priority checks come before "Both feasible" so that subbasins where
# one intervention clearly dominates are not absorbed into "Both."
#
# Thresholds (15%, 20%, 2 km2) are analyst-defined screening cutoffs,
# not scientifically derived values.  They should be stated as such
# in the thesis methods section.
#
# Output: planner/subbasin_screening_map.geojson (EPSG:4326)
# ==========================================================

from pandas.api.types import is_float_dtype, is_integer_dtype

print("\n" + "="*70)
print("PHASE 13 -- EXPORT SUBBASIN SCREENING MAP (GEOJSON)")
print("="*70)


def classify_screening(row):
    """
    Rule-based screening classification for a single subbasin.

    Uses pct_*_gte80 (share of valid suitability area above 80)
    and area_*_gte80_km2 (absolute area above 80) as the main
    decision variables.
    """
    ms = safe_float(row.get("mean_marab_suitability"), None)
    vs = safe_float(row.get("mean_vallerani_suitability"), None)
    pm80 = safe_float(row.get("pct_marab_gte80"), None)
    pv80 = safe_float(row.get("pct_vallerani_gte80"), None)
    am80 = safe_float(row.get("area_marab_gte80_km2"), None)
    av80 = safe_float(row.get("area_vallerani_gte80_km2"), None)
    combined = safe_float(row.get("combined_feasible"), None)

    # 1. Unresolved: no suitability data for either intervention
    if ms is None and vs is None:
        return "Unresolved"

    # 2. Vallerani priority: Vallerani clearly stronger than Marab
    if (
        pv80 is not None and av80 is not None
        and pv80 >= 20
        and av80 >= 2
        and (
            pm80 is None
            or (pv80 - (pm80 or 0)) >= 10
            or (am80 is not None and am80 > 0 and av80 >= 1.5 * am80)
        )
    ):
        return "Vallerani priority"

    # 3. Marab priority: Marab clearly stronger than Vallerani
    if (
        pm80 is not None and am80 is not None
        and pm80 >= 20
        and am80 >= 2
        and (
            pv80 is None
            or ((pm80 or 0) - (pv80 or 0)) >= 10
            or (av80 is not None and av80 > 0 and am80 >= 1.5 * av80)
        )
    ):
        return "Marab priority"

    # 4. Both feasible: both interventions show meaningful high-suitability area
    #    (checked after priority so dominant-intervention cases are not absorbed)
    if (
        combined == 1.0
        and pm80 is not None and pv80 is not None
        and pm80 >= 15
        and pv80 >= 15
    ):
        return "Both feasible"

    # 5. Monitor: moderate or mixed suitability
    return "Monitor"


# Apply classification
enriched["screening_class"] = enriched.apply(classify_screening, axis=1)

class_counts = enriched["screening_class"].value_counts()
print("  Classification summary:")
for cls_name, cnt in class_counts.items():
    print(f"    {cls_name}: {cnt}")

# Columns to carry into the GeoJSON
screening_cols = [
    "SUB", "area_km2", "screening_class",
    "mean_marab_suitability", "mean_vallerani_suitability",
    "pct_marab_gte80", "pct_vallerani_gte80",
    "area_marab_gte80_km2", "area_vallerani_gte80_km2",
    "ratio_marab_gte80", "ratio_vallerani_gte80",
    "combined_feasible",
    "mean_ndvi_delta", "ndvi_trend",
    "mean_annual_precip_mm",
    "active_layers_complete",
]
screening_cols = [c for c in screening_cols if c in enriched.columns]

# Harmonize join key type
enriched["SUB"] = pd.to_numeric(enriched["SUB"], errors="coerce").astype("Int64")
gdf_4326["SUB"] = pd.to_numeric(gdf_4326["SUB"], errors="coerce").astype("Int64")

# Join classification and metrics back to polygon geometries
gdf_out = gdf_4326[["SUB", "geometry"]].merge(
    enriched[screening_cols],
    on="SUB",
    how="left",
    suffixes=("", "_drop"),
)
drop_cols = [c for c in gdf_out.columns if c.endswith("_drop")]
if drop_cols:
    gdf_out = gdf_out.drop(columns=drop_cols)

# Ensure EPSG:4326 for Cesium
if gdf_out.crs is None:
    gdf_out = gdf_out.set_crs("EPSG:4326")
elif gdf_out.crs.to_epsg() != 4326:
    gdf_out = gdf_out.to_crs("EPSG:4326")

# Convert numeric columns to plain Python types for clean GeoJSON
for col in gdf_out.columns:
    if col == "geometry":
        continue
    if is_float_dtype(gdf_out[col]):
        gdf_out[col] = gdf_out[col].apply(
            lambda v: round(float(v), 6) if pd.notna(v) and np.isfinite(v) else None
        )
    elif is_integer_dtype(gdf_out[col]):
        gdf_out[col] = gdf_out[col].apply(
            lambda v: int(v) if pd.notna(v) else None
        )

# Write to planner folder
OUT_SCREENING_GEOJSON = OUT_DIR / "subbasin_screening_map.geojson"
gdf_out.to_file(OUT_SCREENING_GEOJSON, driver="GeoJSON")

print(f"\n  Features: {len(gdf_out)}")
print(f"  CRS: {gdf_out.crs}")
print(f"  Output: {OUT_SCREENING_GEOJSON}")
print(f"  File size: {OUT_SCREENING_GEOJSON.stat().st_size / 1024:.0f} KB")


## 7. Combined suitability candidate overlay

Create a PNG showing pixels where both Marab and Vallerani suitability exceed
the 80% threshold simultaneously.


In [ ]:



print("\n" + "="*70)
print("EXPORT COMBINED CANDIDATE OVERLAY (BOTH >= 80)")
print("="*70)

if MARAB_SUIT_TIF.exists() and VALLERANI_SUIT_TIF.exists():

    # Compute common bounds (same logic as Phase 11)
    west_all, south_all = 999, 999
    east_all, north_all = -999, -999
    for fp in [MARAB_SUIT_TIF, VALLERANI_SUIT_TIF]:
        with rasterio.open(fp) as src:
            w, s, e, n = transform_bounds(src.crs, OUT_CRS, *src.bounds, densify_pts=21)
            west_all = min(west_all, w)
            south_all = min(south_all, s)
            east_all = max(east_all, e)
            north_all = max(north_all, n)

    dst_transform = from_bounds(west_all, south_all, east_all, north_all, OUT_WIDTH, OUT_HEIGHT)

    # Reproject both rasters to the common grid
    reprojected = {}
    for label, tif_path, vmin in [
        ("marab", MARAB_SUIT_TIF, MARAB_VMIN),
        ("vallerani", VALLERANI_SUIT_TIF, VALLERANI_VMIN),
    ]:
        with rasterio.open(tif_path) as src:
            arr = src.read(1).astype(np.float32)
            if src.nodata is not None:
                arr = np.where(arr == src.nodata, np.nan, arr)

            dst_arr = np.full((OUT_HEIGHT, OUT_WIDTH), np.nan, dtype=np.float32)
            reproject(
                source=arr,
                destination=dst_arr,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=dst_transform,
                dst_crs=OUT_CRS,
                src_nodata=np.nan,
                dst_nodata=np.nan,
                resampling=Resampling.nearest,
            )

        # Mark pixels below valid_min as invalid
        dst_arr[~np.isfinite(dst_arr)] = np.nan
        dst_arr[dst_arr < vmin] = np.nan
        reprojected[label] = dst_arr

    # Find pixels where BOTH are >= 80
    marab_above = np.isfinite(reprojected["marab"]) & (reprojected["marab"] >= PRIMARY_THRESHOLD)
    vallerani_above = np.isfinite(reprojected["vallerani"]) & (reprojected["vallerani"] >= PRIMARY_THRESHOLD)
    both_above = marab_above & vallerani_above

    # Also track where at least one is valid (for statistics)
    either_valid = np.isfinite(reprojected["marab"]) | np.isfinite(reprojected["vallerani"])

    # Build RGBA image
    # Use a distinct colour that is clearly different from the individual overlays
    # Marab uses orange (#FF8C00), Vallerani uses purple (#7B68EE)
    # Combined uses teal (#00CED1) for visual distinction
    rgba = np.zeros((OUT_HEIGHT, OUT_WIDTH, 4), dtype=np.uint8)
    rgba[both_above, 0] = 0      # R
    rgba[both_above, 1] = 206    # G
    rgba[both_above, 2] = 209    # B
    rgba[both_above, 3] = 210    # alpha

    out_png = CANDIDATE_OUT_DIR / "combined_candidate_gt80_4326.png"
    Image.fromarray(rgba, mode="RGBA").save(out_png)

    # Statistics
    n_marab_only = int((marab_above & ~vallerani_above).sum())
    n_vallerani_only = int((vallerani_above & ~marab_above).sum())
    n_both = int(both_above.sum())
    n_neither = int((either_valid & ~marab_above & ~vallerani_above).sum())

    print(f"  Marab only >= {PRIMARY_THRESHOLD}:     {n_marab_only:,} pixels")
    print(f"  Vallerani only >= {PRIMARY_THRESHOLD}: {n_vallerani_only:,} pixels")
    print(f"  Both >= {PRIMARY_THRESHOLD}:           {n_both:,} pixels")
    print(f"  Neither >= {PRIMARY_THRESHOLD}:        {n_neither:,} pixels")
    print(f"  Exported: {out_png.name}")

    # Update the manifest JSON with the combined entry
    if CANDIDATE_BOUNDS_JSON.exists():
        with open(CANDIDATE_BOUNDS_JSON, "r", encoding="utf-8") as f:
            manifest = json.load(f)
    else:
        manifest = {
            "crs": OUT_CRS,
            "west": west_all, "south": south_all,
            "east": east_all, "north": north_all,
            "width": OUT_WIDTH, "height": OUT_HEIGHT,
            "thresholds": {}
        }

    if str(PRIMARY_THRESHOLD) not in manifest["thresholds"]:
        manifest["thresholds"][str(PRIMARY_THRESHOLD)] = {}

    manifest["thresholds"][str(PRIMARY_THRESHOLD)]["combined"] = {
        "path": "combined_candidate_gt80_4326.png",
        "west": west_all,
        "south": south_all,
        "east": east_all,
        "north": north_all,
        "pixels_marab_only": n_marab_only,
        "pixels_vallerani_only": n_vallerani_only,
        "pixels_both": n_both,
        "pixels_neither": n_neither,
        "threshold": PRIMARY_THRESHOLD,
    }

    with open(CANDIDATE_BOUNDS_JSON, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"  Manifest updated: {CANDIDATE_BOUNDS_JSON}")

else:
    print("  Both suitability TIFFs are required for the combined overlay.")
    if not MARAB_SUIT_TIF.exists():
        print(f"    Missing: {MARAB_SUIT_TIF}")
    if not VALLERANI_SUIT_TIF.exists():
        print(f"    Missing: {VALLERANI_SUIT_TIF}")

## Summary

This notebook produced the planner-side decision-support files for the dashboard:

- Sub-basin indicator profiles (suitability, NDVI, precipitation for all 429 sub-basins)
- Enriched master JSON with SWAT summaries for the 71 overlapping sub-basins
- Scenario deltas for the SWAT-supported sub-basins
- Screening map classifying sub-basins into four restoration categories
- Combined candidate overlay for dual-intervention zones

Sub-basin screening results (Section 4.5):
- 282 (65.7%) primarily Vallerani-suitable
- 45 (10.5%) suitable for both interventions
- 94 (21.9%) flagged for monitoring
- 8 (1.9%) excluded due to insufficient raster coverage

Thesis references: Section 3.4.3 (methodology), Section 4.5 (screening results),
Section 4.7 (prototype outputs).
